# Read catalog metadata and clean supplied text

Run from `lcats/` or `lcats/notebooks/` with LCATS installed. This offline example
uses one repository RDF fixture (provenance in its fixtures/provenance.json) and
an explicitly temporary catalog. It never downloads or changes a working cache.


In [ ]:
import pathlib
import tarfile
import tempfile
from lcats.gettenberg import catalog
from lcats.gettenberg import cleanup
from lcats.gettenberg import ingestion
from lcats.gettenberg import lifecycle
from lcats.utils import paths

project = paths.find_pyproject_root(pathlib.Path.cwd())
fixture = project / "tests/gettenberg_tests/fixtures/pg84.rdf"


In [ ]:
with tempfile.TemporaryDirectory() as directory:
    root = pathlib.Path(directory)
    manager = lifecycle.CatalogManager(root / "cache")
    print("Before explicit setup:", manager.status()["state"])
    archive = root / "sample.tar.bz2"
    with tarfile.open(archive, "w:bz2", format=tarfile.USTAR_FORMAT) as bundle:
        bundle.add(fixture, arcname=fixture.name)
    candidate = root / "catalog.sqlite"
    ingestion.build_catalog(archive, candidate)
    with catalog.Catalog(candidate) as library:
        book = library.get(84).ebook
        print([title.value.text for title in book.titles])
        print("Listed resources:", len(book.resources))


In [ ]:
text = ("The Project Gutenberg eBook of A Local Example\n\n"
        "*** START OF THE PROJECT GUTENBERG EBOOK A LOCAL EXAMPLE ***\n\n"
        "The travelers waited beside the river.\n\n"
        "*** END OF THE PROJECT GUTENBERG EBOOK A LOCAL EXAMPLE ***\n")
result = cleanup.strip_boilerplate(text)
print(result.text)
print(result.diagnostics)


Listing resource metadata does not download it. To obtain real bytes, follow
[explicit acquisition](../docs/how-to/gettenberg-resources.md),
[routing](../docs/how-to/gettenberg-routing.md) and
[strict decoding](../docs/how-to/gettenberg-decoding.md). For existing catalogs use
`with lifecycle.CatalogManager(cache_dir).open() as library:`; a missing catalog
raises a setup error rather than downloading silently. CLI `lcats gather` retains
smart setup and its approval policy.
